<a href="https://colab.research.google.com/github/MUHAMMAD-RAHEEL-SARWAR/Autism_Biomarker_ML/blob/main/GSE26415_Predicting_Autism_Spectrum_Disorder_Using_Blood_based_Gene_Expression_Signatures_and_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==========================================
# 1. ENVIRONMENT CONFIGURATION & INSTALLS
# ==========================================
!pip install numpy pandas scikit-learn rpy2 joblib --quiet

# Install R's Bioconductor core manager and limma package
import os

# ==========================================
# 2. PYTHON STANDARD & ML IMPORTS
# ==========================================
import urllib.request
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix

# ==========================================
# 3. R-PYTHON BRIDGE INITIALIZATION
# ==========================================
import rpy2.robjects as robjects
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# Activate the automatic pandas-to-R dataframe converter
pandas2ri.activate()

# Install R's Bioconductor core manager and limma package directly within the rpy2 R session
robjects.r("if (!requireNamespace('BiocManager', quietly = TRUE)) install.packages('BiocManager', repos='https://cloud.r-project.org')")
robjects.r("suppressMessages(BiocManager::install('limma', update=FALSE, ask=FALSE))")

# Import the limma library into our active R session
limma = importr('limma')

print("--- Step 1 Complete: Global Environment and Cross-Language Bridge successfully initialized! ---")

(as ‘lib’ is unspecified)







	‘/tmp/Rtmpr6V23s/downloaded_packages’





	‘/tmp/Rtmpr6V23s/downloaded_packages’



--- Step 1 Complete: Global Environment and Cross-Language Bridge successfully initialized! ---


In [5]:
# ==========================================
# STEP 2: DATA ACQUISITION & PREPROCESSING
# ==========================================
import gzip

# The precise NCBI URL path for your dataset
DATA_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE26nnn/GSE26415/matrix/GSE26415_series_matrix.txt.gz"

print("Downloading GSE26415 transcriptomic dataset from NCBI GEO...")
urllib.request.urlretrieve(DATA_URL, "GSE26415_series_matrix.txt.gz")
print("Download complete.")

# 1. Parse expression matrix, programmatically ignoring the GEO metadata headers
print("Parsing expression matrix and aligning phenotypes...")
raw_df = pd.read_csv("GSE26415_series_matrix.txt.gz", sep="\t", compression="gzip", comment="!")
raw_df.set_index("ID_REF", inplace=True)

# 2. Extract clinical patient characteristics from the file metadata headers
with gzip.open("GSE26415_series_matrix.txt.gz", "rt") as f:
    lines = f.readlines()

# Use the columns from the raw_df as the definitive sample IDs
sample_ids = raw_df.columns.tolist()

characteristics = []
# Now iterate to find the characteristics, assuming only one relevant line for 'disease'
for line in lines:
    if line.startswith("!Sample_characteristics_ch1"):
        characteristics = line.strip().split("\t")[1:]
        characteristics = [c.replace('"', '').replace("'", "") for c in characteristics]
        break

# 3. Format clinical metadata mapping
metadata_df = pd.DataFrame({
    "Sample_ID": sample_ids,
    "Diagnosis": characteristics
})
metadata_df.set_index("Sample_ID", inplace=True)

# Map characteristics cleanly to binary targets (ASD = 1, Control = 0)
metadata_df["Class"] = metadata_df["Diagnosis"].apply(
    lambda x: "ASD" if "autism" in x.lower() else "Control"
)

# 4. Final Alignment Matrix construction
raw_df = raw_df[metadata_df.index]
X_raw_all = raw_df.T
y_all = metadata_df["Class"].map({"Control": 0, "ASD": 1}).values

print("\n=========================================================")
print("         DATA ACQUISITION & ALIGNMENT SUCCESS            ")
print("=========================================================")
print(f"Dataset Identifier:     GSE26415")
print(f"Total Patient Samples:  {X_raw_all.shape[0]}")
print(f"Total Genomic Features: {X_raw_all.shape[1]} probes")
print(f"Class Breakdown:        {np.sum(y_all == 1)} ASD, {np.sum(y_all == 0)} Control")
print("=========================================================")

Download complete.
Parsing expression matrix and aligning phenotypes...

         DATA ACQUISITION & ALIGNMENT SUCCESS            
Dataset Identifier:     GSE26415
Total Patient Samples:  84
Total Genomic Features: 19194 probes
Class Breakdown:        21 ASD, 63 Control


In [6]:
# ==========================================
# STEP 3: R-LIMMA STATISTICAL ENGINE
# ==========================================

# 1. Convert our clean Python strings into an R-compatible factor array
diagnosis_vector = ["ASD" if val == 1 else "Control" for val in y_all]
r_diagnosis = robjects.StrVector(diagnosis_vector)

# Push the raw expression dataframe directly into R environment memory space
with (robjects.default_converter + pandas2ri.converter).context():
    robjects.globalenv['r_expression_matrix'] = raw_df

# Store the clean class factor vector inside R
robjects.globalenv['r_diagnosis_factor'] = r_diagnosis

print("Executing R-limma Empirical Bayes differential expression calculations...")

# 2. Run the core limma architecture natively inside R's environment
robjects.r('''
    # Enforce classification array as an explicit R factor variable
    status <- factor(r_diagnosis_factor, levels=c("Control", "ASD"))

    # Construct the structural Design Matrix
    design <- model.matrix(~0 + status)
    colnames(design) <- c("Control", "ASD")

    # Fit the linear modeling parameters across all 19,194 features
    fit <- lmFit(r_expression_matrix, design)

    # Define the precise comparative contrast profile (ASD vs Control Phenotype)
    contrast_matrix <- makeContrasts(ASD_vs_Control = ASD - Control, levels=design)
    fit2 <- contrasts.fit(fit, contrast_matrix)

    # Compute Empirical Bayes shrinkage adjustments for standard errors
    fit2 <- eBayes(fit2)
''')

print("\n=========================================================")
print("          R-LIMMA ENGINE INITIALIZED SUCCESS            ")
print("=========================================================")
print("Linear modeling and Empirical Bayes metrics calculated.")
print("The active R workspace memory is optimized for feature sweeps.")
print("=========================================================")

Executing R-limma Empirical Bayes differential expression calculations...

          R-LIMMA ENGINE INITIALIZED SUCCESS            
Linear modeling and Empirical Bayes metrics calculated.
The active R workspace memory is optimized for feature sweeps.


In [9]:
# =========================================================
# STEP 4 & 5: BALANCED AUTOMATED SWEEP & CLINICAL REPORT
# =========================================================

print("Initializing Class-Balanced Automated Feature Sweep (5 to 50 biomarkers)...\n")

feature_counts = range(5, 51)
best_accuracy = 0
best_feature_count = 0
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scaler_all = StandardScaler()

# 1. Run the sweep using balanced model parameters
for count in feature_counts:
    robjects.r(f'temp_table_auto <- topTable(fit2, coef="ASD_vs_Control", number={count}, sort.by="P")')
    r_df_auto = robjects.globalenv['temp_table_auto']
    with (robjects.default_converter + pandas2ri.converter).context():
        biomarker_df_auto = robjects.conversion.get_conversion().rpy2py(r_df_auto)

    current_probes = list(biomarker_df_auto.index)
    X_auto_scaled = scaler_all.fit_transform(X_raw_all[current_probes])

    # Using an SVM with automated class balancing to defeat the 3-to-1 imbalance
    svm_balanced = SVC(C=1.0, kernel='rbf', class_weight='balanced', random_state=42)
    scores = cross_val_score(svm_balanced, X_auto_scaled, y_all, cv=cv_strategy, scoring='accuracy')
    mean_accuracy = np.mean(scores)

    if mean_accuracy > best_accuracy:
        best_accuracy = mean_accuracy
        best_feature_count = count

# 2. Extract the absolute champion balanced panel
robjects.r(f'final_table <- topTable(fit2, coef="ASD_vs_Control", number={best_feature_count}, sort.by="P")')
final_df_r = robjects.globalenv['final_table']
with (robjects.default_converter + pandas2ri.converter).context():
    final_biomarker_df = robjects.conversion.get_conversion().rpy2py(final_df_r)

champion_probes = list(final_biomarker_df.index)
X_final_scaled = scaler_all.fit_transform(X_raw_all[champion_probes])

# 3. Generate final balanced out-of-fold predictions
production_svm = SVC(C=1.0, kernel='rbf', class_weight='balanced', random_state=42)
cv_predictions = cross_val_predict(production_svm, X_final_scaled, y_all, cv=cv_strategy)

# 4. Compute metrics
tn, fp, fn, tp = confusion_matrix(y_all, cv_predictions).ravel()
final_accuracy = ((tp + tn) / len(y_all)) * 100
final_sensitivity = (tp / (tp + fn)) * 100
final_specificity = (tn / (tn + fp)) * 100

print("=========================================================")
print("  FINAL CLINICAL DIAGNOSTIC REPORT: BALANCED CLASS MODEL ")
print("=========================================================\n")
print(f"Optimal Biomarker Panel Size: {best_feature_count} Probes")
print(f"Identified Biomarkers:        {', '.join(champion_probes)}\n")
print(f"Overall Diagnostic Accuracy:  {final_accuracy:.1f}%")
print(f"Clinical Sensitivity (ASD):  {final_sensitivity:.1f}%")
print(f"Clinical Specificity (Ctrl): {final_specificity:.1f}%")
print(f"Confusion Matrix: True Negative={tn}, False Positive={fp}, False Negative={fn}, True Positive={tp}")
print("\n=========================================================")

# 5. Serialize final balanced assets
joblib.dump(production_svm, 'autism_balanced_model.pkl')
joblib.dump(scaler_all, 'autism_balanced_scaler.pkl')
print("Saved production assets: 'autism_balanced_model.pkl' & 'autism_balanced_scaler.pkl'")

Initializing Class-Balanced Automated Feature Sweep (5 to 50 biomarkers)...

  FINAL CLINICAL DIAGNOSTIC REPORT: BALANCED CLASS MODEL 

Optimal Biomarker Panel Size: 25 Probes
Identified Biomarkers:        A_32_P184330, A_23_P21382, A_24_P912765, A_24_P666340, A_24_P642758, A_32_P190181, A_23_P167464, A_24_P925361, A_32_P2103, A_32_P129810, A_24_P500584, A_32_P14737, A_23_P149050, A_24_P933400, A_32_P179746, A_32_P81173, A_32_P146635, A_32_P136800, A_24_P787914, A_32_P161554, A_32_P159726, A_32_P106315, A_32_P218806, A_32_P90685, A_23_P43425

Overall Diagnostic Accuracy:  82.1%
Clinical Sensitivity (ASD):  90.5%
Clinical Specificity (Ctrl): 79.4%
Confusion Matrix: True Negative=50, False Positive=13, False Negative=2, True Positive=19

Saved production assets: 'autism_balanced_model.pkl' & 'autism_balanced_scaler.pkl'
